# Бейзлайн: классификация эмоций питомцев

Решение задачи с помощью **предобученной CNN** (ResNet18) как фичевыделителя и **логистической регрессии** поверх эмбеддингов.

План:
1. Загрузка данных и настройка препроцессинга
2. Загрузка предобученной ResNet18 и получение эмбеддингов
3. Обучение классификатора на train
4. Предсказание на test и сохранение submission

## 1. Импорты и настройка путей

In [ ]:
from pathlib import Path
import csv
import numpy as np
import pandas as pd
import torch
from torchvision import models, transforms
from PIL import Image
from sklearn.linear_model import LogisticRegression

# Пути к данным (если ноутбук в той же папке, что и data_pets)
ROOT = Path(".").resolve()
DATA = ROOT / "data_pets"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Data folder exists:", DATA.exists())

## 2. Препроцессинг изображений

ResNet обучен на ImageNet, поэтому используем те же нормализацию и размер 224×224.

In [ ]:
IMG_SIZE = 224

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

print("Transform ready: Resize -> ToTensor -> Normalize(ImageNet)")

## 3. Загрузка предобученной ResNet18 (без классификационной головы)

Берём все слои кроме последнего полносвязного — на выходе получаем вектор признаков (эмбеддинг) размерности 512.

In [ ]:
def load_feature_extractor():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    # убираем последний слой (fc) — оставляем только "хвост" для эмбеддингов
    model = torch.nn.Sequential(*list(model.children())[:-1])
    model.eval().to(DEVICE)
    return model

model = load_feature_extractor()
print("ResNet18 feature extractor loaded.")

## 4. Функция получения эмбеддингов по списку путей к картинкам

In [ ]:
def get_embeddings(model, image_paths):
    model.eval()
    embs = []
    with torch.no_grad():
        for path in image_paths:
            img = Image.open(path).convert("RGB")
            x = transform(img).unsqueeze(0).to(DEVICE)
            out = model(x)
            embs.append(out.cpu().numpy().ravel())
    return np.stack(embs)

print("Function get_embeddings defined.")

## 5. Загрузка train-данных и получение эмбеддингов

In [ ]:
train_df = pd.read_csv(DATA / "train.csv")
train_paths = [DATA / "train" / f for f in train_df["image"]]
y_train = train_df["label"].values

print("Train size:", len(train_df))
print("Classes:", sorted(train_df["label"].unique()))
print(train_df["label"].value_counts())

In [ ]:
print("Computing train embeddings (может занять минуту)...")
X_train = get_embeddings(model, train_paths)
print("X_train shape:", X_train.shape)

## 6. Обучение логистической регрессии

In [ ]:
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

train_acc = clf.score(X_train, y_train)
print("Train accuracy:", train_acc)

## 7. Предсказание на test и сохранение submission

In [ ]:
test_df = pd.read_csv(DATA / "test.csv")
test_paths = [DATA / "test" / f for f in test_df["image"]]

print("Test size:", len(test_df))
print("Computing test embeddings...")
X_test = get_embeddings(model, test_paths)
preds = clf.predict(X_test)

In [ ]:
out_path = DATA / "submission_baseline.csv"
with open(out_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["image", "label"])
    for fname, label in zip(test_df["image"], preds):
        w.writerow([fname, label])

print("Saved:", out_path)
print("Sample predictions:")
pd.DataFrame({"image": test_df["image"], "label": preds}).head(10)

## Готово

Файл `submission_baseline.csv` можно загрузить на Kaggle в разделе **Submit Predictions**. При желании отметьте этот сабмит как **Benchmark**, чтобы он отображался на лидерборде как референс для участников.